In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/home/t/terencew/project-cluo/igvf/2023_YR2/multiome/'
samples = list(np.loadtxt(f'{projdir}/txt/samples.txt', dtype=str))
s = samples[0]

In [ ]:
!ls -lt {projdir}/h5ad/rna/raw/filtered.h5ad

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/rna/raw/filtered.h5ad')

In [ ]:
adata

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=10)

In [ ]:
adata

### normalization

In [ ]:
sc.pp.normalize_total(adata,
                      exclude_highly_expressed=True)

# sc.pp.normalize_total(adata, exclude_highly_expressed=True)

In [ ]:
sc.pp.log1p(adata)

In [ ]:
n_top_genes = 3000
sc.pp.highly_variable_genes(adata, n_top_genes=n_top_genes)

In [ ]:
sc.pp.scale(adata, max_value=10)

In [ ]:
n_comps = 50
sc.tl.pca(adata, n_comps=n_comps)

In [ ]:
sc.pl.pca(adata, components=['1,2','3,4','5,6','7,8',
                            '9,10','11,12','13,14',
                             '15,16', '17,18','19,20'],
          wspace=0.3)

In [ ]:
sc.pl.pca(adata, components=['1,2','3,4','5,6','7,8'],
                             color=['n_genes', 'n_genes_by_counts',
                                    'pct_counts_mt', 'total_counts',
                                   'time_num', 'assignment'], wspace=0.3)

In [ ]:
to_keep = 20
sc.pp.neighbors(adata, random_state=0, n_pcs=to_keep)
sc.tl.leiden(adata, resolution=0.5, random_state=0)
sc.tl.umap(adata, random_state=0)

In [ ]:
sc.pl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=['leiden', 'time_num', 'assignment'], wspace=0.3)

### add trajectory score

In [ ]:
traj_genes = pd.read_csv(f'{projdir}/csv/trajectory/IPS.Trajectory.Gene.List.txt', sep='/t', header=0, index_col=0)
traj_genes.shape

In [ ]:
traj_genes.head()

In [ ]:
esc_genes = traj_genes[traj_genes['Class'] == 'ESC']
fibro_genes = traj_genes[traj_genes['Class'] == 'Fibro']
esc_genes.shape, fibro_genes.shape, \
esc_genes.shape[0] + fibro_genes.shape[0], traj_genes.shape

In [ ]:
adata.var['gene_symbol'] = [x.split('.')[0] for x in adata.var.index]

In [ ]:

fibro_mask = adata.var.index.isin(fibro_genes.index)
np.sum(fibro_mask), fibro_genes.index.shape

In [ ]:
# esc_mask = adata.var['gene_symbol'].isin(esc_genes.index)
esc_mask = adata.var.index.isin(esc_genes.index)
np.sum(esc_mask), esc_genes.index.shape

In [ ]:
np.sum(adata[:, esc_mask].X, axis=1).shape

In [ ]:
def score(x):
    return  x - np.min(x) / (np.max(x) - np.min(x))

def get_traj_score(adata, traj_genes):
    fibro_mask = adata.var.index.isin(fibro_genes.index)
    esc_mask = adata.var.index.isin(esc_genes.index)
    esc_scores = score(np.mean(adata[:, esc_mask].X, axis=1))
    fibro_scores = score(np.mean(adata[:, fibro_mask].X, axis=1))
    return esc_scores - fibro_scores

In [ ]:
traj_scores = get_traj_score(adata, traj_genes)
adata.obs['traj_score'] = traj_scores.copy()

In [ ]:
sc.pl.umap(adata, color=['time_num', 'assignment'])

In [ ]:
sc.pl.umap(adata, color=['traj_score'], vmin=-0.5, vmax=0.5, cmap='magma')

In [ ]:
sc.pl.umap(adata, color=['ambient_rna'], vmin=0, vmax=0.5, cmap='magma')

In [ ]:
adata.write_h5ad(f'{projdir}/h5ad/rna/processed/filtered.h5ad')

In [ ]:
small_adata = anndata.AnnData(obs = adata.obs, var=adata.var)
small_adata.obsm['X_pca'] = adata.obsm['X_pca']
small_adata.obsm['X_umap'] = adata.obsm['X_umap']
small_adata.varm['PCs'] = adata.varm['PCs']
small_adata.obsp['distances'] = adata.obsp['distances']
small_adata.obsp['connectivities'] = adata.obsp['connectivities']

In [ ]:
small_adata.write_h5ad(f'{projdir}/h5ad/rna/processed/filtered_small.h5ad')

In [ ]:
pcs = pd.DataFrame(adata.obsm['X_pca'])
pcs.columns = [f'PC_{x}' for x in range(50)]
pcs.index = adata.obs.index
umaps = pd.DataFrame(adata.obsm['X_umap'])
umaps.columns = [f'umap_{x}' for x in range(2)]
umaps.index = adata.obs.index
pcs.to_csv(f'{projdir}/csv/rna/pcs.csv', sep='\t', header=True, index=True)
umaps.to_csv(f'{projdir}/csv/rna/umap.csv', sep='\t', header=True, index=True)

### now run joint RNA/ATAC
run http://localhost:7670/notebooks/project-cluo/igvf/2023_YR2/multiome/ipynb/joint_rna_atac/joint_embedding/01a_subset_rna_atac_to_intersection.ipynb first

In [ ]:
adata

In [ ]:
joint = pd.read_csv(f'{projdir}/csv/qc/joint/joint_qc_filtered.csv', sep='\t', header=0, index_col=0)
joint.shape

In [ ]:
joint_adata = adata[adata.obs.index.isin(joint.index)]

In [ ]:
joint_adata

In [ ]:
# raw_adata = sc.read_h5ad(f'{projdir}/h5ad/rna/raw/filtered.h5ad')
# tmp_adata = raw_adata[raw_adata.obs.index.isin(joint_adata.obs.index)]
# tmp_adata.write_h5ad(f'{projdir}/h5ad/rna/processed/intersect_atac.h5ad')

In [ ]:
n_comps = 50
sc.tl.pca(joint_adata, n_comps=n_comps)

In [ ]:
sc.pl.pca(joint_adata, components=['1,2','3,4','5,6','7,8'],
                             color=['n_genes', 'n_genes_by_counts',
                                    'pct_counts_mt', 'total_counts',
                                   'time_num', 'assignment'], wspace=0.3)

In [ ]:
to_keep = 20
sc.pp.neighbors(joint_adata, n_neighbors=20, random_state=0, n_pcs=to_keep)
sc.tl.leiden(joint_adata, resolution=0.5, random_state=0)
sc.tl.umap(joint_adata, random_state=0)

In [ ]:
sc.pl.umap(joint_adata, color=['time_num', 'assignment'])

In [ ]:
sc.pl.umap(joint_adata, color=['traj_score'], vmin=-0.5, vmax=0.5, cmap='magma')

In [ ]:
sc.pl.umap(joint_adata, color=['ambient_rna'])

In [ ]:
joint_adata

In [ ]:
joint_adata.write_h5ad(f'{projdir}/h5ad/rna/processed/intersect_atac.h5ad')

In [ ]:
small_adata = anndata.AnnData(obs = joint_adata.obs, var=adata.var)
small_adata.obsm['X_pca'] = joint_adata.obsm['X_pca']
small_adata.obsm['X_umap'] = joint_adata.obsm['X_umap']
small_adata.varm['PCs'] = joint_adata.varm['PCs']
small_adata.obsp['distances'] = joint_adata.obsp['distances']
small_adata.obsp['connectivities'] = joint_adata.obsp['connectivities']
small_adata.write_h5ad(f'{projdir}/h5ad/rna/processed/intersect_atac_small.h5ad')

In [ ]:
pcs = pd.DataFrame(joint_adata.obsm['X_pca'])
pcs.columns = [f'PC_{x}' for x in range(50)]
pcs.index = joint_adata.obs.index
umaps = pd.DataFrame(joint_adata.obsm['X_umap'])
umaps.columns = [f'umap_{x}' for x in range(2)]
umaps.index = joint_adata.obs.index

pcs.to_csv(f'{projdir}/csv/rna/intersect_atac_pcs.csv', sep='\t', header=True, index=True)
umaps.to_csv(f'{projdir}/csv/rna/intersect_atac_umap.csv', sep='\t', header=True, index=True)

In [ ]:
!date